# Тест YOLO за ТЗ кейсу 4.5

Що робить: збирає **незалежний test** Anti-UAV410 SKY (54 серії, 21 551 кадр, модель їх не бачила) і val для вибору порогу, підбирає поріг на val, перевіряє на test окремо по наборах (небо, змішані сцени, порожні кадри, синтетичне сонце, SIRST-набори, якщо додані), рахує FPS, експортує TensorRT FP16 і перевіряє, що Recall не впав.

**Перед запуском**
1. Settings: Accelerator **GPU T4 x2** (працює одна), Internet **On**.
2. **Add Input** → модель: Output ноутбука, де вчили (Your Work / Notebook Output), або датасет з `best.pt`. Ноутбук сам знайде `best.pt` у `/kaggle/input`. Можна замість цього дати посилання на Drive у `WEIGHTS_URL`.
3. Необов'язково: Kaggle → Datasets → New Dataset, завантажити всі `sirst_part*.zip` в один датасет, і **Add Input** цей датасет (справжнє сонце, порожнє небо, точкові цілі).
4. **Save Version → Save & Run All**. Займає ~1-1.5 год. Результат: `test_results.zip` в Output.

In [ ]:
# ======= ПАРАМЕТРИ =======
DRIVE_URL = 'https://drive.google.com/file/d/1zsdazmKS3mHaEZWS2BnqbYHPEcIaH5WR/view'   # сирий AntiUAV410.zip (якщо Quota exceeded, посилання на копію)
WEIGHTS = ''        # порожньо = знайти best.pt у /kaggle/input
WEIGHTS_URL = ''    # або посилання на best.pt на Google Drive
FP_BUDGET = 0.7     # FP на кадр, наш запас до вимоги ТЗ <= 1
TRT = True          # експорт TensorRT FP16 і повторна перевірка на test
INT8 = False        # TensorRT INT8 (довше, може губити дрібні цілі)


In [ ]:
# ======= ПАКЕТ КОМАНДИ: builder і маніфест val (кожен 4-й кадр) + test (усі кадри) =======
import os, base64
os.makedirs('/tmp/dl/team_package', exist_ok=True)
open('/tmp/dl/team_package/build_antiuav_yolo.py', 'w').write(r'''#!/usr/bin/env python3
"""Build a single-class 640x512 YOLO dataset from a curated manifest and RAW ZIP.

Only selected image bytes are read. Inputs are read-only. Nonempty output directories
are refused. Exit codes: 0 complete; 1 partial build with row errors; 2 fatal input error.
"""
from __future__ import annotations
import argparse
import csv
import hashlib
import io
import json
import math
import re
import sys
import zipfile
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath

import pandas as pd
from PIL import Image
import yaml
from tqdm import tqdm

VERSION = '1.1.0'
SPLITS = ('train','val','test')
ROLES = ('positive','negative','hard_negative')
REQUIRED = {'sequence','frame_number','image_name','exist','valid_bbox','bbox_x','bbox_y','bbox_w','bbox_h','sample_role','assigned_split'}
ERROR_FIELDS = ['row','sequence','frame_number','image_name','error','detail']

def digest(path: Path) -> str:
    h=hashlib.sha256()
    with path.open('rb') as src:
        for block in iter(lambda:src.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

def boolean(value) -> bool:
    text=str(value).strip().lower()
    if text in ('true','1','1.0'):return True
    if text in ('false','0','0.0'):return False
    raise ValueError(f'Invalid boolean {value!r}')

def output_stem(sequence: str, frame: int) -> str:
    safe=re.sub(r'[^A-Za-z0-9_-]','_',sequence)[:80]
    key=hashlib.sha256(sequence.encode('utf-8')).hexdigest()[:16]
    return f'{safe}__{key}__{frame:08d}'

def load_manifest(path: Path) -> pd.DataFrame:
    df=pd.read_csv(path,dtype={'sequence':str,'image_name':str,'sample_role':str,'assigned_split':str})
    missing=REQUIRED-set(df.columns)
    if missing:raise ValueError(f'Manifest missing columns: {sorted(missing)}')
    if df.empty:raise ValueError('Manifest is empty')
    for col in ['sequence','image_name','sample_role','assigned_split']:
        if df[col].isna().any() or df[col].str.strip().eq('').any():raise ValueError(f'Blank {col}')
    if not set(df.assigned_split)<=set(SPLITS):raise ValueError('Only train/val/test splits are allowed')
    if not set(df.sample_role)<=set(ROLES):raise ValueError('Unknown sample_role')
    for col in ['frame_number','exist']:
        vals=pd.to_numeric(df[col],errors='raise')
        if vals.isna().any() or not vals.map(lambda x:math.isfinite(x) and x==int(x)).all():raise ValueError(f'Invalid integer {col}')
        df[col]=vals.astype('int64')
    if (df.frame_number<1).any() or not set(df.exist)<={0,1}:raise ValueError('Invalid frame_number or exist')
    if df.duplicated(['sequence','frame_number']).any() or df.duplicated(['sequence','image_name']).any():raise ValueError('Duplicate frame keys')
    if (df.groupby('sequence').assigned_split.nunique()>1).any():raise ValueError('Sequence appears in multiple splits')
    if 'recording_group' in df:
        if df.recording_group.isna().any() or df.recording_group.astype(str).str.strip().eq('').any():raise ValueError('Blank recording_group')
        if (df.groupby('recording_group').assigned_split.nunique()>1).any():raise ValueError('Recording group appears in multiple splits')
    for row in df.itertuples():
        if '/' in row.sequence or '\\' in row.sequence or row.sequence in ('.','..'):raise ValueError('Sequence must be a directory component')
        if PurePosixPath(row.image_name).name!=row.image_name or '\\' in row.image_name or not row.image_name.lower().endswith(('.jpg','.jpeg')):raise ValueError('image_name must be a JPEG basename')
    df['valid_bbox']=df.valid_bbox.map(boolean)
    return df.sort_values(['assigned_split','sequence','frame_number']).reset_index(drop=True)

def index_zip(z: zipfile.ZipFile, sequences: set[str]):
    index=defaultdict(list)
    # ZIP central-directory metadata only. No unselected image is opened/extracted.
    for info in tqdm(z.infolist(),desc='Index ZIP',unit='entries'):
        if info.is_dir():continue
        parts=PurePosixPath(info.filename.replace('\\','/')).parts
        if not parts or not parts[-1].lower().endswith(('.jpg','.jpeg')):continue
        hits=sequences.intersection(parts[:-1])
        for seq in hits:index[(seq,parts[-1].lower())].append((info,parts))
    return index

def resolve_member(index, row):
    candidates=index.get((row.sequence,row.image_name.lower()),[])
    if not candidates:raise FileNotFoundError('Selected JPG not found under sequence directory')
    thermal=[c for c in candidates if any(p.lower() in {'infrared','ir','thermal'} for p in c[1][:-1])]
    if thermal:candidates=thermal
    else:
        candidates=[c for c in candidates if not any(p.lower() in {'visible','rgb','color'} for p in c[1][:-1])]
    if len(candidates)>1 and hasattr(row,'original_split'):
        same=[c for c in candidates if str(row.original_split).lower() in [p.lower() for p in c[1][:-1]]]
        if same:candidates=same
    if len(candidates)!=1:raise ValueError(f'Ambiguous or non-thermal JPG match: {len(candidates)} candidates')
    return candidates[0][0]

def make_label(row, width: int, height: int) -> str:
    if row.sample_role in ('negative','hard_negative'):
        if row.exist!=0 or row.valid_bbox:raise ValueError('Negative role contradicts exist/valid_bbox')
        return ''
    if row.exist!=1 or not row.valid_bbox:raise ValueError('Invalid positive bbox or contradictory exist')
    x,y,w,h=(float(getattr(row,f'bbox_{k}')) for k in ('x','y','w','h'))
    if not all(math.isfinite(v) for v in (x,y,w,h)):raise ValueError('Nonfinite bbox')
    if w<=0 or h<=0 or x<0 or y<0 or x+w>width or y+h>height:raise ValueError(f'BBox outside {width}x{height}: {(x,y,w,h)}')
    norm=((x+w/2)/width,(y+h/2)/height,w/width,h/height)
    if not all(0<=v<=1 for v in norm):raise ValueError('Normalized bbox outside [0,1]')
    return '0 '+' '.join(f'{v:.10f}' for v in norm)+'\n'

def preflight(source, manifest, df, out) -> int:
    """Validate every selected reference using ZIP metadata; no JPEG payloads read."""
    errors=[]; seen=set(); stems=set(); image_bytes=0
    try:
        with zipfile.ZipFile(source,'r') as z:
            index=index_zip(z,set(df.sequence))
            for number,row in enumerate(tqdm(df.itertuples(index=False),total=len(df),desc='Check references',unit='frames'),start=2):
                try:
                    make_label(row,640,512)
                    info=resolve_member(index,row)
                    if info.header_offset in seen:raise ValueError('Same RAW member referenced twice')
                    stem=output_stem(row.sequence,row.frame_number)
                    if stem in stems:raise ValueError('Output name collision')
                    if info.flag_bits & 1:raise ValueError('Encrypted selected ZIP member')
                    if info.file_size==0:raise ValueError('Empty selected JPEG member')
                    if info.compress_type not in {zipfile.ZIP_STORED,zipfile.ZIP_DEFLATED,zipfile.ZIP_BZIP2,zipfile.ZIP_LZMA}:raise ValueError('Unsupported ZIP compression')
                    seen.add(info.header_offset);stems.add(stem);image_bytes+=info.file_size
                except (ValueError,OSError) as exc:
                    errors.append({'row':number,'sequence':row.sequence,'frame_number':row.frame_number,'image_name':row.image_name,'error':type(exc).__name__,'detail':str(exc)})
    except (OSError,zipfile.BadZipFile,RuntimeError) as exc:
        errors.append({'row':'','sequence':'','frame_number':'','image_name':'','error':type(exc).__name__,'detail':'Fatal archive error: '+str(exc)})
    with (out/'preflight_errors.csv').open('w',encoding='utf-8',newline='') as dest:
        writer=csv.DictWriter(dest,fieldnames=ERROR_FIELDS);writer.writeheader();writer.writerows(errors)
    report={'builder_version':VERSION,'status':'metadata_pass' if not errors else 'failed','source':str(source),'source_bytes':source.stat().st_size,'manifest_sha256':digest(manifest),'expected_frames':len(df),'matched_frames':len(seen),'errors':len(errors),'selected_image_bytes':image_bytes,'selected_image_gib':round(image_bytes/(1024**3),3),'expected_output_files':2*len(df)+3,'by_split':df.assigned_split.value_counts().to_dict(),'by_role':df.sample_role.value_counts().to_dict(),'limitations':['ZIP metadata only: JPEG dimensions, decoding, CRC and EXIF are checked during full build.','Image bytes exclude labels, filesystem allocation overhead and report files. Allow additional free space.','No annotation-completeness or cross-sequence image-duplicate audit performed.']}
    (out/'preflight_report.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
    print(f"{report['status'].upper()}: {len(seen)}/{len(df)} references, {len(errors)} errors; images {report['selected_image_gib']} GiB. Report: {out/'preflight_report.json'}")
    return 0 if not errors else 1

def run(args) -> int:
    source=args.source.resolve();manifest=args.manifest.resolve();out=args.output.resolve()
    if not source.is_file() or not manifest.is_file():raise ValueError('Source ZIP and manifest must be existing files')
    if out.exists() and (not out.is_dir() or any(out.iterdir())):raise ValueError('Output must be absent or an empty directory; refusing overwrite')
    df=load_manifest(manifest)
    out.mkdir(parents=True,exist_ok=True)
    if getattr(args,'check_only',False):return preflight(source,manifest,df,out)
    report={'builder_version':VERSION,'source':str(source),'source_bytes':source.stat().st_size,'source_sha256':digest(source) if args.hash_source else None,'manifest':str(manifest),'manifest_sha256':digest(manifest),'expected_frames':len(df),'dimensions':[640,512],'class_names':{'0':'thermal_target'},'status':'building','exported_frames':0,'errors':0,'by_split':{},'by_role':{},'notes':['Strict 640x512; no resizing, clipping, or silent annotation repair.','Positive boxes invalid or out of bounds are skipped, never converted to negative.','Thermal directory preferred; ambiguous matches skipped.','Only selected JPEG payloads read; ZIP central directory indexed.']}
    errors=[];counts=Counter();roles=Counter();seen_members=set();seen_output=set()
    for split in SPLITS:
        for category in ('images','labels'):(out/category/split).mkdir(parents=True,exist_ok=True)
    try:
        with zipfile.ZipFile(source,'r') as z:
            index=index_zip(z,set(df.sequence))
            for number,row in enumerate(tqdm(df.itertuples(index=False),total=len(df),desc='Build YOLO',unit='frames'),start=2):
                image_path=label_path=None
                try:
                    # Validate label semantics before opening an image, including invalid flags.
                    label=make_label(row,640,512)
                    info=resolve_member(index,row)
                    if info.header_offset in seen_members:raise ValueError('Same RAW member referenced twice')
                    raw=z.read(info)
                    with Image.open(io.BytesIO(raw)) as im:
                        if im.format!='JPEG':raise ValueError('Payload is not JPEG')
                        if im.size!=(640,512):raise ValueError(f'Expected 640x512, got {im.size}')
                        if im.mode not in ('L','RGB'):raise ValueError(f'Unsupported image mode {im.mode}')
                        if im.getexif().get(274,1)!=1:raise ValueError('Nontrivial EXIF orientation; coordinate interpretation requires review')
                        im.load() # decode completely; corrupt/truncated JPEG is an error
                    stem=output_stem(row.sequence,row.frame_number)
                    if stem in seen_output:raise ValueError('Output name collision')
                    image_path=out/'images'/row.assigned_split/(stem+'.jpg')
                    label_path=out/'labels'/row.assigned_split/(stem+'.txt')
                    with image_path.open('xb') as dest:dest.write(raw)
                    with label_path.open('x',encoding='utf-8',newline='\n') as dest:dest.write(label)
                    seen_output.add(stem);seen_members.add(info.header_offset)
                    counts[row.assigned_split]+=1;roles[row.sample_role]+=1
                except (ValueError,OSError,zipfile.BadZipFile,RuntimeError,EOFError) as exc:
                    # Only paths created by this row, under an initially empty output, are cleaned.
                    for target in (image_path,label_path):
                        if target is not None and target.is_file():target.unlink()
                    errors.append({'row':number,'sequence':row.sequence,'frame_number':row.frame_number,'image_name':row.image_name,'error':type(exc).__name__,'detail':str(exc)})
    except (OSError,zipfile.BadZipFile,RuntimeError) as exc:
        errors.append({'row':'','sequence':'','frame_number':'','image_name':'','error':type(exc).__name__,'detail':'Fatal archive error: '+str(exc)})
    with (out/'build_errors.csv').open('w',encoding='utf-8',newline='') as dest:
        writer=csv.DictWriter(dest,fieldnames=ERROR_FIELDS);writer.writeheader();writer.writerows(errors)
    yaml_data={'path':out.as_posix(),'train':'images/train','val':'images/val','test':'images/test','names':{0:'thermal_target'}}
    (out/'data.yaml').write_text(yaml.safe_dump(yaml_data,sort_keys=False,allow_unicode=True),encoding='utf-8')
    report.update(exported_frames=sum(counts.values()),errors=len(errors),by_split={s:counts[s] for s in SPLITS},by_role={r:roles[r] for r in ROLES},status='complete' if not errors and sum(counts.values())==len(df) else 'partial')
    report['file_counts']={s:{'images':len(list((out/'images'/s).glob('*.jpg'))),'labels':len(list((out/'labels'/s).glob('*.txt')))} for s in SPLITS}
    (out/'build_report.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
    print(f"{report['status'].upper()}: {report['exported_frames']}/{len(df)} frames, {len(errors)} errors. Report: {out/'build_report.json'}")
    return 0 if report['status']=='complete' else 1

def main() -> int:
    p=argparse.ArgumentParser(description=__doc__,formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument('--source',required=True,type=Path,help='Read-only AntiUAV410_RAW.zip')
    p.add_argument('--manifest',required=True,type=Path,help='selected_frames.csv or .csv.gz')
    p.add_argument('--output',required=True,type=Path,help='New or empty output directory')
    p.add_argument('--hash-source',action='store_true',help='Optionally stream entire ZIP to compute SHA-256; adds a full-file read')
    p.add_argument('--check-only',action='store_true',help='Check all selected ZIP paths and label semantics without reading JPEG payloads; write preflight reports only. Use a separate empty output directory.')
    args=p.parse_args()
    if args.check_only and args.hash_source:p.error('--hash-source cannot be combined with metadata-only --check-only')
    try:return run(args)
    except (ValueError,OSError,KeyError) as exc:
        print(f'FATAL: {exc}',file=sys.stderr);return 2
if __name__=='__main__':raise SystemExit(main())
''')
open('/tmp/dl/team_package/man_test.csv.gz', 'wb').write(base64.b64decode('...'))
open('/kaggle/working/eval_tz.py', 'w').write(r'''"""Оцінка YOLO за ТЗ кейсу 4.5 на незалежному тесті.

Правило: влучання = центр рамки в рамці цілі +-3 px; Recall лише для цілей зі сторонами 5-50 px;
FP рахується на кожному наборі окремо. Поріг підбирається на val, на test лише перевіряється.
"""
import os, glob, json, time
import numpy as np, cv2

TOL = 3
CONFS = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.6]


# ---------- набори ----------
def yolo_items(root, split, meta=None, only=None):
    """Кадри YOLO-датасету. meta: {stem: dict(scene=..., seq=...)}; only: функція-фільтр по meta."""
    items = []
    for p in sorted(glob.glob(f"{root}/images/{split}/*.jpg")):
        n = os.path.basename(p)[:-4]; m = (meta or {}).get(n, {})
        if only and not only(n, m): continue
        g = []
        lf = f"{root}/labels/{split}/{n}.txt"
        if os.path.exists(lf):
            for l in open(lf).read().split("\n"):
                if l.strip():
                    _, cx, cy, w, h = map(float, l.split()); w, h = w * 640, h * 512
                    g.append((cx * 640 - w / 2, cy * 512 - h / 2, w, h))
        items.append(dict(name=n, seq=m.get("seq", n.split("__")[0]), path=p, gts=g))
    return items


def pad640(img):
    """Кадр будь-якого розміру -> 640x512: більший зменшуємо, менший доповнюємо медіаною (як у skydet)."""
    h, w = img.shape[:2]; s = min(1.0, 640 / w, 512 / h)
    if s < 1: img = cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA); h, w = img.shape[:2]
    out = np.full((512, 640), int(np.median(img)), np.uint8); x0, y0 = (640 - w) // 2, (512 - h) // 2
    out[y0:y0 + h, x0:x0 + w] = img
    return out, x0, y0, s


def mask_items(d):
    """Набори skydet (SIRST/IRSTD): images/, masks/, list.txt. Цілі = компоненти маски."""
    items = []
    for n in open(f"{d}/list.txt").read().split():
        p = next(iter(glob.glob(f"{d}/images/{n}.*")), None)
        if p is None: continue
        mp = next(iter(glob.glob(f"{d}/masks/{n}.*")), None)
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE); _, x0, y0, s = pad640(img)
        g = []
        if mp:
            k, _, st, _ = cv2.connectedComponentsWithStats((cv2.imread(mp, 0) > 127).astype(np.uint8), 8)
            g = [(st[i, 0] * s + x0, st[i, 1] * s + y0, st[i, 2] * s, st[i, 3] * s) for i in range(1, k)]
        items.append(dict(name=n, seq=os.path.basename(d), path=p, gts=g, pad=True))
    return items


def load(it):
    img = cv2.imread(it["path"], cv2.IMREAD_GRAYSCALE)
    if it.get("pad") or img.shape != (512, 640): img = pad640(img)[0]
    return cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)


# ---------- передбачення і метрики ----------
def predict(model, items, batch=16, half=False):
    out = []
    for i in range(0, len(items), batch):
        rs = model.predict([load(it) for it in items[i:i + batch]], imgsz=[512, 640], conf=0.01, max_det=20,
                           verbose=False, batch=batch, **({"half": True} if half else {}))
        for r in rs:
            b = r.boxes
            out.append(np.hstack([b.xyxy.cpu().numpy(), b.conf.cpu().numpy()[:, None]]).astype(np.float32))
    return out


def ok_size(w, h): return min(w, h) >= 5 and max(w, h) <= 50


def keep(p, conf, topk, margin):
    p = p[p[:, 4] >= conf]
    if margin: p = p[(p[:, 0] >= margin) & (p[:, 1] >= margin) & (p[:, 2] <= 640 - margin) & (p[:, 3] <= 512 - margin)]
    return p[np.argsort(-p[:, 4])][:topk]


def frame_eval(p, gts):
    """-> (кількість FP, множина знайдених цілей)"""
    fp, hit = 0, set()
    for x0, y0, x1, y1, c in p:
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
        h = [j for j, (x, y, w, hh) in enumerate(gts) if x - TOL <= cx <= x + w + TOL and y - TOL <= cy <= y + hh + TOL]
        fp += not h; hit.update(h)
    return fp, hit


def score(items, preds, cfg):
    conf, topk, margin = cfg["conf"], cfg["topk"], cfg["margin"]
    fps, tp, n, by = [], 0, 0, {}
    for it, p in zip(items, preds):
        fp, hit = frame_eval(keep(p, conf, topk, margin), it["gts"]); fps.append(fp)
        for j, (x, y, w, h) in enumerate(it["gts"]):
            if not ok_size(w, h): continue
            k = "5-8" if max(w, h) <= 8 else "9-16" if max(w, h) <= 16 else "17-32" if max(w, h) <= 32 else "33-50"
            a = by.setdefault(k, [0, 0]); a[1] += 1; n += 1
            if j in hit: a[0] += 1; tp += 1
    fps = np.array(fps) if fps else np.zeros(1)
    return dict(frames=len(items), targets=n, recall=round(tp / n, 4) if n else None,
                fp_per_frame=round(float(fps.mean()), 4), fp_max=int(fps.max()),
                frames_fp_gt1=round(float((fps > 1).mean()), 4), frames_fp_ge1=round(float((fps >= 1).mean()), 4),
                recall_by_size={k: round(a[0] / a[1], 3) for k, a in sorted(by.items())})


def tune(sets, budget, topks):
    """sets: {назва: (items, preds)}. Найбільший Recall на першому наборі, FP <= budget на КОЖНОМУ наборі."""
    best = None
    for topk in topks:
        for margin in [0, 2, 4]:
            for conf in CONFS:
                cfg = dict(conf=conf, topk=topk, margin=margin)
                r = {k: score(it, pr, cfg) for k, (it, pr) in sets.items()}
                if any(x["fp_per_frame"] > budget for x in r.values()): continue
                rec = next(iter(r.values()))["recall"]
                if best is None or rec > best[0] + 1e-9: best = (rec, cfg, r)
    return best


# ---------- розбір помилок ----------
def seq_table(items, preds, cfg):
    t = {}
    for it, p in zip(items, preds):
        fp, hit = frame_eval(keep(p, **cfg), it["gts"])
        a = t.setdefault(it["seq"], dict(sequence=it["seq"], frames=0, fp=0, targets=0, missed=0))
        a["frames"] += 1; a["fp"] += fp
        for j, (x, y, w, h) in enumerate(it["gts"]):
            if ok_size(w, h): a["targets"] += 1; a["missed"] += j not in hit
    return sorted(t.values(), key=lambda a: -(a["fp"] + a["missed"]))


def draw(it, p, cfg, path):
    img = load(it)
    for x, y, w, h in it["gts"]:
        cv2.rectangle(img, (int(x), int(y)), (int(x + w), int(y + h)), (255, 255, 0), 1)
    for x0, y0, x1, y1, c in keep(p, **cfg):
        fp, _ = frame_eval(np.array([[x0, y0, x1, y1, c]]), it["gts"])
        col = (0, 0, 255) if fp else (0, 255, 0)
        cv2.rectangle(img, (int(x0), int(y0)), (int(x1), int(y1)), col, 1)
        cv2.putText(img, f"{c:.2f}", (int(x0), max(int(y0) - 3, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, col, 1)
    cv2.imwrite(path, img)


def gallery(items, preds, cfg, out, n=12):
    """n кадрів із найупевненішими FP і n пропусків, не більше 2 кадрів з однієї серії."""
    os.makedirs(out, exist_ok=True); fps, miss = [], []
    for i, (it, p) in enumerate(zip(items, preds)):
        k = keep(p, **cfg); fp, hit = frame_eval(k, it["gts"])
        if fp: fps.append((-float(k[:, 4].max()), i))
        if any(ok_size(w, h) and j not in hit for j, (x, y, w, h) in enumerate(it["gts"])): miss.append((i,))
    html = ["<html><meta charset='utf-8'><body style='background:#111;color:#eee;font:14px sans-serif'>",
            "<p>Блакитна рамка: розмітка. Зелена: влучання. Червона: хибне спрацювання.</p>"]
    for title, lst in [("Хибні спрацювання", sorted(fps)), ("Пропуски", miss[::max(1, len(miss) // (n * 4))])]:
        html.append(f"<h2>{title}</h2>"); per, k = {}, 0
        for e in lst:
            it = items[e[-1]]
            if per.get(it["seq"], 0) >= 2: continue
            per[it["seq"]] = per.get(it["seq"], 0) + 1; k += 1
            f = f"{'fp' if title[0] == 'Х' else 'miss'}_{k:02d}.jpg"; draw(it, preds[e[-1]], cfg, f"{out}/{f}")
            html.append(f"<div style='display:inline-block;margin:6px'><img src='{f}' width=480><br>{it['name']}</div>")
            if k >= n: break
    open(f"{out}/gallery.html", "w").write("\n".join(html))


# ---------- швидкість ----------
def speed(model, items, cfg, half=False, n=200):
    """Увесь конвеєр на кадрі в пам'яті, batch 1: підготовка, модель, NMS, наші фільтри."""
    imgs = [load(it) for it in items[:: max(1, len(items) // 50)][:50]]
    kw = {"half": True} if half else {}
    for i in range(20): model.predict(imgs[i % len(imgs)], imgsz=[512, 640], conf=cfg["conf"], verbose=False, **kw)
    t = []
    for i in range(n):
        t0 = time.perf_counter()
        b = model.predict(imgs[i % len(imgs)], imgsz=[512, 640], conf=cfg["conf"], verbose=False, **kw)[0].boxes
        keep(np.hstack([b.xyxy.cpu().numpy(), b.conf.cpu().numpy()[:, None]]), **cfg)
        t.append((time.perf_counter() - t0) * 1000)
    return dict(ms_mean=round(float(np.mean(t)), 2), ms_p95=round(float(np.percentile(t, 95)), 2), fps=round(1000 / float(np.mean(t)), 1))
''')
print('записано')

In [ ]:
# ======= ДАНІ =======
!nvidia-smi -L
!pip install -q ultralytics "pandas>=2.2" "Pillow>=10" "PyYAML>=6" tqdm
import os, re, glob, sys, json, zipfile, shutil, subprocess
import numpy as np, pandas as pd, cv2
DL = '/tmp/dl'; PKG = '/tmp/dl/team_package'; os.makedirs(DL, exist_ok=True)
def fetch(url, out):
    # Великі файли Google Drive віддає через сторінку "не вдалося перевірити на віруси".
    # Качаємо напряму з drive.usercontent.google.com з confirm=t, а якщо прийшла сторінка, беремо з неї форму і пробуємо ще раз.
    import requests
    if os.path.exists(out) and os.path.getsize(out) > 1 << 20: print('вже є', out); return
    fid = (re.search(r'/d/([\w-]{20,})', url) or re.search(r'id=([\w-]{20,})', url)).group(1)
    s, u, params = requests.Session(), 'https://drive.usercontent.google.com/download', {'id': fid, 'export': 'download', 'confirm': 't'}
    for _ in range(3):
        r = s.get(u, params=params, stream=True, timeout=120)
        if 'text/html' not in r.headers.get('Content-Type', ''):
            total, done, mark = int(r.headers.get('Content-Length', 0)), 0, 0
            with open(out, 'wb') as f:
                for chunk in r.iter_content(8 << 20):
                    f.write(chunk); done += len(chunk)
                    if done - mark > 1 << 30: mark = done; print(f'  {done / 2**30:.1f} / {total / 2**30:.1f} ГБ', flush=True)
            print(f'скачано {out}: {done / 2**30:.2f} ГБ'); return
        html = r.text
        form = re.search(r'<form[^>]+action="([^"]+)"', html)
        inputs = dict(re.findall(r'<input[^>]*name="([^"]+)"[^>]*value="([^"]*)"', html))
        if not (form and inputs): break
        u, params = form.group(1).replace('&amp;', '&'), inputs
    t = re.findall(r'<title>(.*?)</title>', html, re.S)
    raise SystemExit('Google Drive не віддав файл: ' + (t[0].strip() if t else '?') + '. Перевірте доступ "Усі, хто має посилання".')

RAW = f'{DL}/AntiUAV410.zip'
local = [f for f in glob.glob('/kaggle/input/**/*.zip', recursive=True) if os.path.getsize(f) > 2**30 and 'anti' in os.path.basename(f).lower()]
if local: RAW = local[0]; print('архів з Input:', RAW)
else: fetch(DRIVE_URL, RAW)
assert zipfile.is_zipfile(RAW), 'DRIVE_URL має вести на zip-архів Anti-UAV410'

SRC = '/tmp/sky_test'
if not os.path.exists(f'{SRC}/data.yaml'):
    shutil.rmtree('/tmp/sky_check', ignore_errors=True); shutil.rmtree(SRC, ignore_errors=True)
    r = subprocess.run(f'python {PKG}/build_antiuav_yolo.py --source {RAW} --manifest {PKG}/man_test.csv.gz --output /tmp/sky_check --check-only', shell=True)
    if r.returncode != 0: raise SystemExit('Архів не збігається з маніфестом, див. /tmp/sky_check/preflight_errors.csv')
    subprocess.run(f'python {PKG}/build_antiuav_yolo.py --source {RAW} --manifest {PKG}/man_test.csv.gz --output {SRC}', shell=True)
    print('помилок збірки:', len(pd.read_csv(f'{SRC}/build_errors.csv')))

# Сцена кожного кадру з маніфесту (AIR = чисте небо, MIXED = небо з хмарами/будинками/лісом на краю, GROUND)
sys.path.insert(0, PKG); sys.path.insert(0, '/kaggle/working')
from build_antiuav_yolo import output_stem
import eval_tz as E
man = pd.read_csv(f'{PKG}/man_test.csv.gz', dtype={'sequence': str})
META = {output_stem(r.sequence, int(r.frame_number)): dict(seq=r.sequence, scene=r.scene) for r in man.itertuples()}

# Синтетичне сонце на кадрах test (інший seed, ніж у навчанні): кожен 10-й кадр з ціллю + усі порожні
def add_sun(img, boxes, rng):
    h, w = img.shape
    r = rng.uniform(8, 30); sig = rng.uniform(1.0, 3.0) * r
    for _ in range(200):
        if boxes and rng.random() < 0.5:                       # половина сонць поруч із ціллю (40-120 px)
            bx, by = boxes[rng.integers(len(boxes))]; ang = rng.uniform(0, 2 * np.pi); d = rng.uniform(40, 120) + r
            cx, cy = bx + d * np.cos(ang), by + d * np.sin(ang)
        else:
            cx, cy = rng.uniform(r, w - r), rng.uniform(r, h * 0.6)
        if not (r <= cx < w - r and r <= cy < h - r): continue
        if boxes and min(np.hypot(bx - cx, by - cy) for bx, by in boxes) < 40 + r: continue
        break
    yy, xx = np.mgrid[:h, :w]; d = np.hypot(xx - cx, yy - cy)
    halo = np.where(d <= r, 1.0, np.exp(-((d - r) ** 2) / (2 * sig ** 2)))
    out = img.astype(np.float32) + (255 - img.astype(np.float32)) * halo * rng.uniform(0.6, 1.0)
    out[d <= r] = 255
    return np.clip(out, 0, 255).astype(np.uint8)

SUN = '/tmp/sun_test'
if not os.path.exists(SUN):
    os.makedirs(f'{SUN}/images/test'); os.makedirs(f'{SUN}/labels/test'); rng = np.random.default_rng(1)
    files = sorted(glob.glob(f'{SRC}/images/test/*.jpg'))
    for i, p in enumerate(files):
        n = os.path.basename(p)[:-4]; lab = open(f'{SRC}/labels/test/{n}.txt').read()
        if lab.strip() and i % 10: continue
        img = cv2.imread(p, 0); H, W = img.shape
        centers = [(float(l.split()[1]) * W, float(l.split()[2]) * H) for l in lab.splitlines() if l.strip()]
        cv2.imwrite(f'{SUN}/images/test/{n}.jpg', add_sun(img, centers, rng), [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{SUN}/labels/test/{n}.txt', 'w').write(lab)

# SIRST-набори skydet, якщо додано sirst_part*.zip (усі частини в один датасет)
for f in glob.glob('/kaggle/input/**/sirst_*.zip', recursive=True): zipfile.ZipFile(f).extractall('/tmp/sirst')
for l in glob.glob('/kaggle/input/**/list.txt', recursive=True):   # Kaggle сам розпаковує zip: зливаємо частини в одну папку
    d = os.path.dirname(l); shutil.copytree(d, '/tmp/sirst/' + os.path.basename(d), dirs_exist_ok=True)
SIRST = {os.path.basename(os.path.dirname(l)): os.path.dirname(l) for l in glob.glob('/tmp/sirst/**/list.txt', recursive=True)}
print('SIRST:', {k: len(glob.glob(v + '/images/*')) for k, v in SIRST.items()})

VAL = E.yolo_items(SRC, 'val', META)
TEST = E.yolo_items(SRC, 'test', META)
SETS = {'test_all': TEST,
        'test_AIR': [it for it in TEST if META.get(it['name'], {}).get('scene') == 'AIR'],
        'test_MIXED': [it for it in TEST if META.get(it['name'], {}).get('scene') == 'MIXED'],
        'test_empty': [it for it in TEST if not it['gts']],
        'test_sun_synth': E.yolo_items(SUN, 'test', META)}
for k in ['sky_targets', 'no_targets', 'sun_real']:
    if k in SIRST: SETS['sirst_' + k] = E.mask_items(SIRST[k])
print('val', len(VAL), '|', {k: len(v) for k, v in SETS.items()})

In [ ]:
# ======= МОДЕЛЬ =======
from ultralytics import YOLO
import logging; logging.getLogger('ultralytics').setLevel(logging.ERROR)   # без сотень попереджень у лозі
if WEIGHTS_URL.startswith('http'): WEIGHTS = '/tmp/best.pt'; fetch(WEIGHTS_URL, WEIGHTS)
if not WEIGHTS:
    found = sorted(glob.glob('/kaggle/input/**/best.pt', recursive=True)); print('знайдено:', found)
    assert found, 'Не знайшов best.pt: Add Input з моделлю або WEIGHTS_URL'
    WEIGHTS = found[0]
shutil.copy(WEIGHTS, '/kaggle/working/best.pt'); WEIGHTS = '/kaggle/working/best.pt'
model = YOLO(WEIGHTS)
NAME = os.path.basename(str(model.model.yaml.get('yaml_file', 'yolo'))).replace('.yaml', '')
print('модель:', NAME, round(sum(p.numel() for p in model.model.parameters()) / 1e6, 1), 'M параметрів')

## Поріг на val, перевірка на test
Два варіанти порогу, обидва підібрані на val (FP ≤ FP_BUDGET і на всіх кадрах val, і на порожніх кадрах val):
- **strict**: лише одна найупевненіша рамка на кадр (top-1). Тоді на жодному кадрі не буде більше 1 FP. На кадрі Anti-UAV410 не більше одної цілі.
- **avg**: до 5 рамок на кадр, обмеження лише на середнє FP.

`margin` = не брати рамки, що торкаються краю кадру (px).

In [ ]:
import time
t0 = time.time()
PV = E.predict(model, VAL)
val_sets = {'val': (VAL, PV), 'val_empty': ([it for it in VAL if not it['gts']], [p for it, p in zip(VAL, PV) if not it['gts']])}
CFG = {}
for mode, topks in [('strict', [1]), ('avg', [1, 2, 3, 5])]:
    b = E.tune(val_sets, FP_BUDGET, topks)
    if b is None: b = (0, dict(conf=0.6, topk=1, margin=4), None); print(mode, 'УВАГА: FP > бюджету навіть при conf 0.6')
    CFG[mode] = b[1]; print(mode, b[1], {k: (v['recall'], v['fp_per_frame']) for k, v in (b[2] or {}).items()})
PRED = {k: E.predict(model, v) for k, v in SETS.items()}
RES = {mode: {k: E.score(SETS[k], PRED[k], cfg) for k in SETS} for mode, cfg in CFG.items()}
rows = [dict(config=m, set=k, **{c: v[c] for c in ['frames', 'targets', 'recall', 'fp_per_frame', 'fp_max', 'frames_fp_gt1']}) for m in RES for k, v in RES[m].items()]
TABLE = pd.DataFrame(rows); print(f'{(time.time() - t0) / 60:.0f} хв'); TABLE

In [ ]:
# Recall за розміром цілі (більша сторона, px) і серії з найбільшою кількістю помилок (strict)
print(pd.DataFrame({k: v['recall_by_size'] for k, v in RES['strict'].items()}).T)
SEQ = pd.DataFrame(E.seq_table(TEST, PRED['test_all'], CFG['strict']))
SEQ['scene'] = SEQ.sequence.map(man.drop_duplicates('sequence').set_index('sequence').scene)
OUT = '/kaggle/working/test_results'; os.makedirs(OUT, exist_ok=True)
E.gallery(TEST, PRED['test_all'], CFG['strict'], f'{OUT}/gallery')
if 'test_sun_synth' in SETS: E.gallery(SETS['test_sun_synth'], PRED['test_sun_synth'], CFG['strict'], f'{OUT}/gallery_sun', n=8)
SEQ.head(15)

## Швидкість і TensorRT
Увесь конвеєр на кадрі в пам'яті, batch 1: підготовка кадру, модель, NMS, наші фільтри. T4 ≈ 2080 SUPER за швидкістю в FP16 (2080 SUPER трохи швидша), тож цифри T4 близькі до фінальних.

In [ ]:
SPEED = {'pytorch_fp32': E.speed(model, TEST, CFG['strict']), 'pytorch_fp16': E.speed(model, TEST, CFG['strict'], half=True)}
KEY = [k for k in ['test_all', 'test_empty', 'test_sun_synth', 'sirst_sun_real', 'sirst_no_targets'] if k in SETS]
for kind in (['fp16'] + (['int8'] if INT8 else [])) if TRT else []:
    try:
        kw = dict(half=True) if kind == 'fp16' else dict(int8=True, data=f'{SRC}/data.yaml', fraction=0.05)
        eng = YOLO(WEIGHTS).export(format='engine', imgsz=[512, 640], batch=1, device=0, **kw)
        dst = f'/kaggle/working/best_{kind}.engine'; shutil.move(eng, dst); m = YOLO(dst, task='detect')
        SPEED['trt_' + kind] = E.speed(m, TEST, CFG['strict'])
        for k in KEY:
            r = E.score(SETS[k], E.predict(m, SETS[k], batch=1), CFG['strict'])
            RES.setdefault('strict_trt_' + kind, {})[k] = r
            print(kind, k, 'recall', r['recall'], '(PyTorch', RES['strict'][k]['recall'], ') FP', r['fp_per_frame'])
    except Exception as e:
        print('TensorRT', kind, 'не вдався:', repr(e)[:300])
pd.DataFrame(SPEED).T

In [ ]:
# ======= ЗБЕРЕГТИ =======
rep = dict(model=NAME, weights=WEIGHTS, gpu=(__import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else 'cpu'), fp_budget=FP_BUDGET, configs=CFG,
           results=RES, speed=SPEED, notes=['Поріг підібрано на val, test не використовувався для вибору.',
           'Влучання: центр рамки в рамці цілі +-3 px; Recall лише для цілей 5-50 px.',
           'test_sun_synth зроблено тим самим генератором сонця, що й у навчанні: оцінка оптимістична. Справжнє сонце: sirst_sun_real.'])
json.dump(rep, open(f'{OUT}/test_metrics.json', 'w'), indent=1, ensure_ascii=False)
TABLE.to_csv(f'{OUT}/summary.csv', index=False); SEQ.to_csv(f'{OUT}/sequences.csv', index=False)
# сирі передбачення test (conf >= 0.01): рядок = [номер кадру, x0, y0, x1, y1, conf], щоб потім перепідібрати пороги без моделі
P = [np.hstack([np.full((len(p), 1), i, np.float32), p]) for i, p in enumerate(PRED['test_all'])]
np.savez_compressed(f'{OUT}/pred_test.npz', names=np.array([it['name'] for it in TEST]), preds=np.vstack(P))
shutil.make_archive('/kaggle/working/test_results', 'zip', OUT)
print(json.dumps(dict(model=NAME, configs=CFG, speed=SPEED), ensure_ascii=False, indent=1))
print(TABLE.to_string(index=False))